In [28]:
import gymnasium as gym;
import numpy as np;
import time;
import pygame;
from enum import Enum

In [29]:
# Enum for Map Sizes
MAP_SIZE = Enum('Size', [('SMALL', "4x4"), ('MEDIUM', "8x8"), ('LARGE', "16x16")])

# Environment Control Variables
global_map_size = MAP_SIZE.SMALL.value 

In [30]:
# create a environment
env = gym.make("FrozenLake-v1",
                map_name = global_map_size,
                is_slippery = False);

In [31]:
# making a Q table
num_states = env.observation_space.n;
num_actions = env.action_space.n;

print("Number of states:", num_states);
print("Number of actions:", num_actions);

q_table = np.zeros((num_states, num_actions));

print("Q-table shape:", q_table.shape);

Number of states: 16
Number of actions: 4
Q-table shape: (16, 4)


### Q-Learning Hyperparameters

**α (Learning Rate)**  
Controls how much newly acquired information overwrites old knowledge.  
- High α → learns quickly but can be unstable  
- Low α → learns slowly but is more stable  

**γ (Discount Factor)**  
Determines how much future rewards are valued compared to immediate rewards.  
- γ close to 1 → future rewards matter a lot  
- γ close to 0 → agent focuses on immediate rewards  

**ε (Exploration Rate)**  
Defines the probability of taking a random action instead of the greedy (best-known) action.  
- High ε → more exploration  
- Low ε → more exploitation


In [32]:
# Learning hyperparameters
alpha = 0.1      # learning rate
gamma = 0.99     # discount factor
epsilon = 1.0    # exploration rate
epsilon_min = 0.01  # minimum exploration rate
epsilon_decay = 0.995 # exploration decay rate

num_episodes = 2000
max_steps_per_episode = 100

In [33]:
# Training loop
for episode in range(num_episodes):
    state, info = env.reset(seed=42)
    done = False

    for step in range(max_steps_per_episode):
        # ε-greedy action selection
        if np.random.random() < epsilon:
            action = env.action_space.sample()  # explore
        else:
            action = np.argmax(q_table[state])  # exploit

        next_state, reward, terminated, truncated, info = env.step(action)

        # Q-learning update
        if terminated:
            target = reward
        else:
            target = reward + gamma * np.max(q_table[next_state])

        q_table[state][action] += alpha * (target - q_table[state][action])
        state = next_state

        if terminated or truncated:
            break

    # Decay exploration
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

env.close()
print("Training complete.");
print("Q-table:");
print(q_table);

Training complete.
Q-table:
[[8.69779105e-01 9.50990050e-01 6.81066272e-01 9.01587688e-01]
 [8.66463947e-01 0.00000000e+00 2.60027295e-03 1.37621947e-01]
 [7.66217823e-02 0.00000000e+00 0.00000000e+00 9.23765290e-06]
 [1.34414611e-04 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.14845879e-01 9.60596010e-01 0.00000000e+00 8.61855209e-01]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [8.79935991e-01 0.00000000e+00 9.70299000e-01 8.67152530e-01]
 [8.67205423e-01 9.80100000e-01 7.56047565e-01 0.00000000e+00]
 [9.49339112e-01 9.89996053e-02 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 9.05045400e-01 9.90000000e-01 9.01487535e-01]
 [7.84815248e-01 9.20015554e-01 1.00000000e+00 6.66381386e-01]
 [0.00000000e+00 0.00000000

In [ ]:
# Testing the learned policy

# Seed the action space RNG once if you need to apply same sequence of steps every time
# env.action_space.seed(42)

# update environment render mode to human for visualization
env = gym.make("FrozenLake-v1",
                map_name = global_map_size,
                is_slippery = False,
                render_mode="human")

state, info = env.reset(seed=42)

env_replay_count = 0

while (not terminated or not truncated) and env_replay_count < 5:
    # Pick greedy action
    action = np.argmax(q_table[state])

    # Step in the environment and get next state
    next_state, reward, terminated, truncated, info = env.step(action)

    print(f"Action: {action}, New state: {next_state}, Reward: {reward}, Terminated: {terminated}, Truncated: {truncated}")

    # Update current state
    state = next_state

    # Slow down visualization
    time.sleep(0.5)

    # Reset if episode ended
    if terminated or truncated:
        state, info = env.reset(seed=42)
        env_replay_count += 1

env.close()


Action: 1, New state: 4, Reward: 0, Terminated: False, Truncated: False
Action: 1, New state: 8, Reward: 0, Terminated: False, Truncated: False
Action: 2, New state: 9, Reward: 0, Terminated: False, Truncated: False
Action: 1, New state: 13, Reward: 0, Terminated: False, Truncated: False
Action: 2, New state: 14, Reward: 0, Terminated: False, Truncated: False
Action: 2, New state: 15, Reward: 1, Terminated: True, Truncated: False
Action: 1, New state: 4, Reward: 0, Terminated: False, Truncated: False
Action: 1, New state: 8, Reward: 0, Terminated: False, Truncated: False
Action: 2, New state: 9, Reward: 0, Terminated: False, Truncated: False
Action: 1, New state: 13, Reward: 0, Terminated: False, Truncated: False
Action: 2, New state: 14, Reward: 0, Terminated: False, Truncated: False
Action: 2, New state: 15, Reward: 1, Terminated: True, Truncated: False
Action: 1, New state: 4, Reward: 0, Terminated: False, Truncated: False
Action: 1, New state: 8, Reward: 0, Terminated: False, Trunc

: 